# Agentic RFT: Tool Calling

Train `Alibaba qwen3.6-35b-a3b` version `1` with the exact successful 10/10 Tool Calling RFT contract in Sweden Central. This notebook ends after terminal monitoring and service reward metrics; it does not deploy or invoke the fine-tuned model.

> Actual results may vary by model version, data, configuration, region availability, and service conditions.


## 1. Setup and paid-operation boundary

Copy this module's `.env.template` to a module-local `.env`, configure your own `FOUNDRY_PROJECT_ENDPOINT` and HTTPS `RFT_TOOL_SERVER_URL`, and sign in with `az login`. Your project must support agentic RFT for the selected model. Run from the cookbook directory or its notebooks directory.

For a protected tool, inject the same high-entropy `FOUNDRY_TOOL_AUTH_TOKEN` into the hosted catalog server and the notebook environment (or ignored local `.env`). Both the contract-check POST and the service method use its Authorization header. Never put the credential in the URL, display the live method/headers, or enable HTTP/SDK wire-debug logs.

Data, tool, and grader checks run locally. Executing the training cell validates your remote tool, authenticates, uploads data, and submits or resumes one paid job.


In [ ]:
import csv
import hashlib
import importlib.util
import json
import math
import os
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from dotenv import load_dotenv

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent] if cwd.name == "notebooks" else [cwd]
ROOT = next((p for p in candidates if (p / "data" / "manifest.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the cookbook or its notebooks directory.")
load_dotenv(ROOT / ".env", override=False)

# Load this demo's helper directly; no cross-recipe imports or sys.path changes.
runtime_spec = importlib.util.spec_from_file_location(
    "agentic_rft_runtime", ROOT / "src" / "agentic_rft_runtime.py",
)
if runtime_spec is None or runtime_spec.loader is None:
    raise ImportError("The demo-local runtime helper is missing.")
runtime = importlib.util.module_from_spec(runtime_spec)
runtime_spec.loader.exec_module(runtime)
load_tool_headers = runtime.load_tool_headers
validate_remote_tool = runtime.validate_remote_tool
submit_or_resume_job = runtime.submit_or_resume_job
redact_tool_secrets = runtime.redact_tool_secrets

PROJECT_ENDPOINT = os.getenv("FOUNDRY_PROJECT_ENDPOINT", "").strip()
REGION = "Sweden Central"
MODEL = "qwen3.6-35b-a3b"
MODEL_VERSION = "1"
TRAINING_TYPE = "globalStandard"
TRAIN_PATH = ROOT / "data" / "tool-calling-train.jsonl"
VALIDATION_PATH = ROOT / "data" / "tool-calling-validation.jsonl"
EXPECTED_TRAIN_SHA256 = "11acebfc227c48fee9dd608734ac81b45544bec76c7b814c01e905783b0d6bc4"
EXPECTED_VALIDATION_SHA256 = "7835cef9a161a1a0311d6b140c782ba60e644aec54641a74746ceab1cdc23517"
TERMINAL_STATUSES = {"succeeded", "failed", "cancelled"}
OUTPUTS = ROOT / "outputs"
STATE_PATH = OUTPUTS / "submission-state.json"
TOOL_SERVER_URL = os.getenv("RFT_TOOL_SERVER_URL", "").strip()
print({"model": MODEL, "model_version": MODEL_VERSION, "training_type": TRAINING_TYPE})

## 2. Provenance and exact guards

The bundled tool-calling training and validation files preserve the original demonstration records, with checkout line endings normalized to LF for portable integrity checks and upload. Exactly 10 rows per split, fixed canonical SHA-256 hashes, the original `search_catalog` function schema, messages, and reference answers are preserved. The manifest uses source descriptions instead of private checkout paths.


In [ ]:
EXPECTED_TOOL = {
    "type": "function",
    "function": {
        "name": "search_catalog",
        "description": (
            "Search the product catalog and return matching jacket items with sku, "
            "name, and price."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text.",
                },
                "top_k": {
                    "type": "integer",
                    "description": "Maximum number of items to return.",
                    "default": 3,
                },
            },
            "required": ["query"],
        },
    },
}

MULTI_GRADER = {
    "type": "multi",
    "name": "strict_partial_credit",
    "invalid_grade": 0.0,
    "graders": {
        "exact": {
            "type": "string_check",
            "name": "exact",
            "operation": "eq",
            "input": "{{sample.output_text}}",
            "reference": "{{item.reference_answer}}",
        },
        "fuzzy": {
            "type": "text_similarity",
            "name": "fuzzy",
            "evaluation_metric": "fuzzy_match",
            "input": "{{sample.output_text}}",
            "reference": "{{item.reference_answer}}",
        },
    },
    "calculate_output": "0.9 * exact + 0.1 * fuzzy",
}


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes().replace(b"\r\n", b"\n")).hexdigest()

def validate_dataset_contract() -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    train_rows = read_jsonl(TRAIN_PATH)
    validation_rows = read_jsonl(VALIDATION_PATH)
    assert len(train_rows) == 10
    assert len(validation_rows) == 10
    assert sha256(TRAIN_PATH) == EXPECTED_TRAIN_SHA256
    assert sha256(VALIDATION_PATH) == EXPECTED_VALIDATION_SHA256
    for split, rows in (("train", train_rows), ("validation", validation_rows)):
        for index, row in enumerate(rows):
            assert row["tools"] == [EXPECTED_TOOL], (
                f"{split} row {index} changed the search_catalog contract"
            )
            assert row["messages"]
            assert row["reference_answer"]
    return train_rows, validation_rows

train_rows, validation_rows = validate_dataset_contract()
import importlib.util

tool_spec = importlib.util.spec_from_file_location("customer_catalog_tool", ROOT / "tools" / "catalog_server.py")
tool_module = importlib.util.module_from_spec(tool_spec)
tool_spec.loader.exec_module(tool_module)
for row in train_rows + validation_rows:
    for arguments in row.get("reference_tool_args", []):
        items = tool_module.execute_search_catalog(**arguments)["items"]
        assert row["reference_answer"] in {item["sku"] for item in items}
assert MULTI_GRADER["calculate_output"] == "0.9 * exact + 0.1 * fuzzy"
assert MULTI_GRADER["invalid_grade"] == 0.0
print({"training_rows": len(train_rows), "validation_rows": len(validation_rows),
       "training_sha256": sha256(TRAIN_PATH), "validation_sha256": sha256(VALIDATION_PATH),
       "grader": MULTI_GRADER})


## 3. Visible training recipe

The service multi-grader uses 90% exact string match plus 10% fuzzy similarity, with invalid outputs scored as zero. Tool episodes allow at most 10 steps. Evaluation runs every 3 steps with 5 samples, compute multiplier 1.0, and medium reasoning effort.


In [ ]:
HYPERPARAMETERS = {
    "eval_interval": 3,
    "eval_samples": 5,
    "compute_multiplier": 1.0,
    "reasoning_effort": "medium",
}
# The secret stays in memory and goes only to the intended tool/service request.
# Never display METHOD or TOOL_HEADERS, or enable HTTP/SDK wire-debug logging.
TOOL_HEADERS = load_tool_headers()
remote_tool = {"name": "search_catalog", "server_url": TOOL_SERVER_URL}
if TOOL_HEADERS:
    remote_tool["headers"] = TOOL_HEADERS.copy()
METHOD = {
    "type": "reinforcement",
    "reinforcement": {
        "grader": MULTI_GRADER,
        "tools": [remote_tool],
        "max_episode_steps": 10,
        "hyperparameters": HYPERPARAMETERS,
    },
}
print({"model": MODEL, "version": MODEL_VERSION, "hyperparameters": HYPERPARAMETERS,
       "max_episode_steps": 10, "tool_auth_configured": bool(TOOL_HEADERS)})

## 4. Validate the remote tool, upload, and submit once

Executing this cell crosses the paid-operation boundary. The authenticated request envelope and expected catalog response are checked before submission, without following redirects. Host the reusable tool at an HTTPS URL reachable by the training service; its catalog, query behavior, and function-call envelope must match the checked-in data.

The demo-local helper atomically writes ignored `outputs/submission-state.json` with the project, redacted method, method/credential fingerprint, canonical data hashes, and cleanup-compatible file/job IDs. It records a pending operation before **each** upload/job POST and saves **each** returned ID before polling or further work. SDK POST retries are disabled. No credential or secret-bearing request payload is saved.

Rerunning resumes saved progress. Pending uploads reconcile by unique content-addressed filename and verified remote bytes; an empty list never authorizes another POST. An unidentified job POST stops until you reconcile it in Foundry and set `FOUNDRY_RECONCILED_TOOL_RFT_JOB_ID` locally to the verified job ID. Mismatched recipes, old state formats, and changed credentials fail closed. Never clear a pending marker or archive state simply to retry. See the [demo recovery guide](../README.md#durable-submission-and-recovery), including stale-lock handling, before manual recovery.


In [ ]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

if not PROJECT_ENDPOINT:
    raise RuntimeError("Set FOUNDRY_PROJECT_ENDPOINT in your .env or process environment.")
validate_dataset_contract()
# Use exactly the header mapping sent in METHOD; no URL credentials or redirects.
validate_remote_tool(TOOL_SERVER_URL, headers=TOOL_HEADERS)
print("This workflow uses paid cloud resources and may incur charges.")

job = None
credential = DefaultAzureCredential(process_timeout=120)
project = None
client = None
try:
    project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
    client = project.get_openai_client(max_retries=0)
    job = submit_or_resume_job(
        client, state_path=STATE_PATH, project_endpoint=PROJECT_ENDPOINT,
        model=MODEL, model_version=MODEL_VERSION, training_type=TRAINING_TYPE,
        method=METHOD, training_path=TRAIN_PATH, validation_path=VALIDATION_PATH,
        expected_training_sha256=EXPECTED_TRAIN_SHA256,
        expected_validation_sha256=EXPECTED_VALIDATION_SHA256,
        reconciled_job_id=os.getenv("FOUNDRY_RECONCILED_TOOL_RFT_JOB_ID", ""),
    )
except BaseException:
    # Closing local clients never cancels a remote operation or clears its state.
    for resource in (client, project, credential):
        if resource is not None:
            try:
                resource.close()
            except Exception:
                pass
    raise RuntimeError(
        "Submission stopped. Keep outputs/submission-state.json and reconcile its "
        "saved IDs/pending operation before resuming. No POST is automatically repeated."
    ) from None
print({"job_status": str(job.status), "submission_state_saved": True})

## 5. Monitor with a timeout and download service reward metrics

A failed or cancelled job raises an error. A timeout or interrupted local client leaves the saved job available to resume; it does not cancel training. Rerun the training cell to reconstruct clients and retrieve that same job. Events are redacted before output or persistence; do not display the live job method or raw transport errors.

Cancellation and file deletion are separate, confirmed operations described in [Post-training and cleanup](../../../POST_TRAINING.md). The saved state retains the verified project endpoint and each file/job ID, including partial uploads. Reconcile pending operations before cleanup; delete only explicitly confirmed, unshared resources.


In [ ]:
def event_dict(event: Any) -> dict[str, Any]:
    if hasattr(event, "to_dict"):
        record = event.to_dict()
    else:
        record = json.loads(event.to_json())
    return redact_tool_secrets(record, METHOD)

def wait_for_job(
    client: Any,
    job_id: str,
    events_path: Path,
    poll_seconds: float,
 ) -> Any:
    seen: set[str] = set()
    records: list[dict[str, Any]] = []
    deadline = time.monotonic() + 21_600
    while True:
        job = client.fine_tuning.jobs.retrieve(job_id)
        print(f"{datetime.now(timezone.utc).isoformat()} job: {job.status}")
        for event in reversed(list(client.fine_tuning.jobs.list_events(job_id, limit=100))):
            record = event_dict(event)
            key = json.dumps(record, sort_keys=True, default=str)
            if key not in seen:
                seen.add(key)
                records.append(record)
                print(f"  {record.get('level', 'info')}: {record.get('message', '')}")
        events_path.write_text(
            json.dumps(records, indent=2, default=str),
            encoding="utf-8",
        )
        if str(job.status).casefold() in TERMINAL_STATUSES:
            if str(job.status).casefold() != "succeeded":
                raise RuntimeError(f"Training ended as {job.status}; inspect the job in Foundry.")
            return job
        if time.monotonic() >= deadline:
            raise TimeoutError("Job exceeded 21600 seconds; rerun the training cell to resume.")
        time.sleep(poll_seconds)

def download_result_files(client: Any, job: Any, run_dir: Path) -> list[Path]:
    paths: list[Path] = []
    for file_id in job.result_files or []:
        response = client.files.content(file_id)
        content = getattr(response, "content", None)
        if content is None and hasattr(response, "read"):
            content = response.read()
        if not isinstance(content, bytes):
            raise RuntimeError("A result file did not return bytes.")
        path = run_dir / f"{file_id}.csv"
        path.write_bytes(content)
        paths.append(path)
    return paths

def finite_float(value: Any) -> float | None:
    if value in (None, ""):
        return None
    parsed = float(value)
    if not math.isfinite(parsed):
        raise ValueError("The service metric is not finite.")
    return parsed

def summarize_curve(rows: list[dict[str, str]], column: str) -> dict[str, Any]:
    points = [
        {"step": int(row["step"]), "value": finite_float(row.get(column))}
        for row in rows
        if finite_float(row.get(column)) is not None
    ]
    if not points:
        return {"metric": column, "points": 0, "trend": "unavailable", "values": []}
    start = points[0]["value"]
    end = points[-1]["value"]
    delta = end - start
    best = max(points, key=lambda point: point["value"])
    return {
        "metric": column,
        "points": len(points),
        "start": start,
        "end": end,
        "delta": delta,
        "best": best["value"],
        "best_step": best["step"],
        "trend": "improving" if delta > 0 else "declining" if delta < 0 else "flat",
        "values": points,
    }

if job is None or client is None:
    raise RuntimeError("Run the training cell to submit or resume the saved job first.")
try:
    run_dir = OUTPUTS / job.id
    run_dir.mkdir(parents=True, exist_ok=True)
    job = wait_for_job(client, job.id, run_dir / "events.json", 60)
    result_paths = download_result_files(client, job, run_dir)
    metric_rows = []
    for path in result_paths:
        with path.open("r", encoding="utf-8-sig", newline="") as handle:
            metric_rows.extend(csv.DictReader(handle))
    train_curve = summarize_curve(metric_rows, "train_mean_reward")
    validation_curve = summarize_curve(metric_rows, "full_valid_mean_reward")
    if not train_curve["values"] and not validation_curve["values"]:
        raise RuntimeError("No training or validation reward metrics returned by the service.")
    summary = {"status": job.status, "model": MODEL, "model_version": MODEL_VERSION,
               "training_reward": train_curve, "validation_reward": validation_curve}
    (run_dir / "run-summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(json.dumps(summary, indent=2))
except BaseException:
    raise RuntimeError(
        "Monitoring or metrics collection stopped. The saved job is not cancelled; "
        "rerun the training cell to reconstruct clients and resume, or follow cleanup guidance."
    ) from None
finally:
    for resource in (client, project, credential):
        try:
            resource.close()
        except Exception:
            print("A local client did not close cleanly; no remote cancellation was requested.")

## 6. Training-only conclusion

Use only the current job's service `train_mean_reward` and `full_valid_mean_reward` curves. The prior reward of `1.0` is reference context, not evidence for this run. The notebook does not deploy or invoke the trained model.

> Actual results may vary by model version, data, configuration, region availability, and service conditions.
